# S7-03: 건축공학 구조 에이전트 — 구조 검토, 멀티에이전트, 설계 최적화
**구조공학 도메인 전용 에이전트 실습**

## 학습 목표
- RC 구조 부재 자동 검토 에이전트를 구축한다
- Designer → Reviewer → Reporter 멀티에이전트 파이프라인을 구현한다
- Evaluator-Optimizer 패턴으로 설계 최적화 에이전트를 구현한다
- 모든 실습이 건축공학 구조 도메인에 직접 적용된다

## 사전 준비
1. `.env` 파일에 API 키 설정:
```
ANTHROPIC_API_KEY="YOUR_API_KEY_HERE"
```

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경 설정
import json
import math
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

print("환경 설정 완료")

---
## Exercise 1: RC 구조 부재 검토 에이전트

### 문제
다음 **구조 계산 도구**를 사용하여 RC 기둥/보를 자율적으로 검토하는 에이전트를 구현하세요.

**도구 5개:**
1. `calc_axial_ratio(Pu_kN, fck_MPa, b_mm, h_mm)` → 축력비
2. `calc_rebar_ratio(As_mm2, b_mm, d_mm)` → 철근비
3. `calc_moment_capacity(b_mm, d_mm, As_mm2, fck_MPa, fy_MPa)` → 공칭 모멘트 강도
4. `calc_shear_capacity(b_mm, d_mm, fck_MPa)` → 콘크리트 전단 강도
5. `check_kds_requirement(check_type)` → KDS 기준 조항 조회

**에이전트 시스템 프롬프트:**
```
당신은 KDS 14 20 기준에 따라 RC 구조물을 검토하는 에이전트입니다.
1. 먼저 부재의 기본 특성(축력비, 철근비)을 확인하세요
2. 그 다음 강도 검토(모멘트, 전단)를 수행하세요
3. 관련 KDS 기준을 확인하세요
4. 모든 결과를 종합하여 최종 판정을 내리세요
```

### 테스트 입력
```
C1 기둥: 500x500mm, fck=27MPa, fy=400MPa,
주근 8-D25 (As=3973mm2), 유효깊이 d=440mm,
설계 축력 Pu=3000kN, 설계 모멘트 Mu=200kN·m, 설계 전단력 Vu=150kN
```

### 기대 출력
에이전트가 자율적으로 도구를 호출하여 모든 검토 항목을 수행하고, 종합 판정을 제시한다.

In [ ]:
# TODO: RC 구조 부재 검토 에이전트를 구현하세요

# 1. 도구 5개 정의 (tools 리스트)
# YOUR CODE HERE

# 2. 도구 실행 함수 (실제 구조 계산 로직 포함)
# YOUR CODE HERE

# 3. 에이전트 루프
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

structural_tools = [
    {
        "name": "calc_axial_ratio",
        "description": "RC 기둥의 축력비를 KDS 14 20 기준으로 계산한다. 축력비 = Pu / (0.85 * fck * Ag)",
        "input_schema": {
            "type": "object",
            "properties": {
                "Pu_kN": {"type": "number", "description": "설계 축력 (kN)"},
                "fck_MPa": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
                "b_mm": {"type": "number", "description": "단면 폭 (mm)"},
                "h_mm": {"type": "number", "description": "단면 높이 (mm)"}
            },
            "required": ["Pu_kN", "fck_MPa", "b_mm", "h_mm"]
        }
    },
    {
        "name": "calc_rebar_ratio",
        "description": "RC 부재의 철근비 rho = As / (b * d)를 계산한다. KDS 14 20 기준: 0.01 <= rho <= 0.08",
        "input_schema": {
            "type": "object",
            "properties": {
                "As_mm2": {"type": "number", "description": "철근 총 단면적 (mm2)"},
                "b_mm": {"type": "number", "description": "단면 폭 (mm)"},
                "d_mm": {"type": "number", "description": "유효 깊이 (mm)"}
            },
            "required": ["As_mm2", "b_mm", "d_mm"]
        }
    },
    {
        "name": "calc_moment_capacity",
        "description": "RC 부재의 설계 휨 강도 phi*Mn을 계산한다. 등가 사각형 응력 블록 방법 사용.",
        "input_schema": {
            "type": "object",
            "properties": {
                "b_mm": {"type": "number", "description": "단면 폭 (mm)"},
                "d_mm": {"type": "number", "description": "유효 깊이 (mm)"},
                "As_mm2": {"type": "number", "description": "인장 철근 면적 (mm2)"},
                "fck_MPa": {"type": "number", "description": "콘크리트 강도 (MPa)"},
                "fy_MPa": {"type": "number", "description": "철근 항복강도 (MPa)"}
            },
            "required": ["b_mm", "d_mm", "As_mm2", "fck_MPa", "fy_MPa"]
        }
    },
    {
        "name": "calc_shear_capacity",
        "description": "RC 부재의 콘크리트 전단 강도 Vc를 계산한다. Vc = (1/6) * sqrt(fck) * b * d",
        "input_schema": {
            "type": "object",
            "properties": {
                "b_mm": {"type": "number", "description": "단면 폭 (mm)"},
                "d_mm": {"type": "number", "description": "유효 깊이 (mm)"},
                "fck_MPa": {"type": "number", "description": "콘크리트 강도 (MPa)"}
            },
            "required": ["b_mm", "d_mm", "fck_MPa"]
        }
    },
    {
        "name": "check_kds_requirement",
        "description": "KDS 설계기준의 특정 검토 항목에 대한 기준값과 조항을 조회한다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "check_type": {
                    "type": "string",
                    "description": "검토 항목: axial_ratio, rebar_ratio, shear, tie_spacing"
                }
            },
            "required": ["check_type"]
        }
    }
]

def execute_structural_tool(name: str, inputs: dict) -> str:
    """구조 계산 도구를 실행한다."""
    if name == "calc_axial_ratio":
        Pu_N = inputs["Pu_kN"] * 1000
        Ag = inputs["b_mm"] * inputs["h_mm"]
        fck = inputs["fck_MPa"]
        ratio = Pu_N / (0.85 * fck * Ag)
        limit = 0.80
        return json.dumps({
            "axial_ratio": round(ratio, 4),
            "limit": limit,
            "status": "OK" if ratio <= limit else "NG",
            "Ag_mm2": Ag,
            "formula": f"Pu/(0.85*fck*Ag) = {Pu_N:.0f}/(0.85*{fck}*{Ag}) = {ratio:.4f}"
        })

    elif name == "calc_rebar_ratio":
        rho = inputs["As_mm2"] / (inputs["b_mm"] * inputs["d_mm"])
        return json.dumps({
            "rebar_ratio": round(rho, 5),
            "min_limit": 0.01,
            "max_limit": 0.08,
            "status": "OK" if 0.01 <= rho <= 0.08 else "NG",
            "formula": f"As/(b*d) = {inputs['As_mm2']:.0f}/({inputs['b_mm']}*{inputs['d_mm']}) = {rho:.5f}"
        })

    elif name == "calc_moment_capacity":
        As = inputs["As_mm2"]
        fy = inputs["fy_MPa"]
        fck = inputs["fck_MPa"]
        b = inputs["b_mm"]
        d = inputs["d_mm"]
        a = (As * fy) / (0.85 * fck * b)  # 등가 응력 블록 깊이
        Mn = As * fy * (d - a / 2) / 1e6  # kN*m
        phi = 0.85  # 강도감소계수 (휨)
        phi_Mn = phi * Mn
        return json.dumps({
            "a_mm": round(a, 1),
            "Mn_kNm": round(Mn, 1),
            "phi_Mn_kNm": round(phi_Mn, 1),
            "phi": phi,
            "formula": f"a = As*fy/(0.85*fck*b) = {a:.1f}mm, Mn = As*fy*(d-a/2) = {Mn:.1f} kN*m"
        })

    elif name == "calc_shear_capacity":
        fck = inputs["fck_MPa"]
        b = inputs["b_mm"]
        d = inputs["d_mm"]
        Vc = (1/6) * math.sqrt(fck) * b * d / 1000  # kN
        phi_Vc = 0.75 * Vc
        return json.dumps({
            "Vc_kN": round(Vc, 1),
            "phi_Vc_kN": round(phi_Vc, 1),
            "phi": 0.75,
            "formula": f"Vc = (1/6)*sqrt({fck})*{b}*{d}/1000 = {Vc:.1f} kN"
        })

    elif name == "check_kds_requirement":
        kds_data = {
            "axial_ratio": {"section": "KDS 14 20 21 4.3", "limit": "축력비 <= 0.80", "note": "초과 시 단면 확대 필요"},
            "rebar_ratio": {"section": "KDS 14 20 21 4.3.1", "limit": "0.01 <= rho <= 0.08", "note": "기둥 주근 기준"},
            "shear": {"section": "KDS 14 20 22 4.5", "limit": "phi*Vn >= Vu", "note": "Vn = Vc + Vs"},
            "tie_spacing": {"section": "KDS 14 20 22 4.3.3", "limit": "min(b/2, 48*db_tie, 3/4*s_main)", "note": "내진 시 더 엄격"}
        }
        ct = inputs["check_type"]
        if ct in kds_data:
            return json.dumps(kds_data[ct], ensure_ascii=False)
        return json.dumps({"error": f"'{ct}' 검토 항목을 찾을 수 없습니다"})

    return json.dumps({"error": f"Unknown tool: {name}"})


def structural_review_agent(query: str, max_turns: int = 12) -> str:
    """RC 구조 부재 검토 에이전트."""
    system = (
        "당신은 KDS 14 20 기준에 따라 RC 구조물을 검토하는 전문 에이전트입니다.\n"
        "1. 먼저 부재의 기본 특성(축력비, 철근비)을 확인하세요\n"
        "2. 그 다음 강도 검토(모멘트, 전단)를 수행하세요\n"
        "3. 관련 KDS 기준을 확인하세요\n"
        "4. 모든 결과를 종합하여 최종 판정을 내리세요\n"
        "도구의 계산 결과를 그대로 사용하고, 결과를 표로 정리하세요."
    )
    messages = [{"role": "user", "content": query}]
    tool_call_count = 0

    for turn in range(max_turns):
        response = client.messages.create(
            model=model, max_tokens=2500,
            system=system, tools=structural_tools, messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            print(f"\n[Agent] 총 {tool_call_count}회 도구 호출 후 완료")
            return final

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_call_count += 1
                result = execute_structural_tool(block.name, block.input)
                parsed = json.loads(result)
                status = parsed.get("status", "")
                print(f"  [{tool_call_count}] {block.name} → {status} | {result[:80]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        messages.append({"role": "user", "content": tool_results})

    return "최대 반복 횟수 도달"


# 실행
print("=== RC 구조 부재 검토 에이전트 ===")
review_result = structural_review_agent(
    "C1 기둥: 500x500mm, fck=27MPa, fy=400MPa, "
    "주근 8-D25 (As=3973mm2), 유효깊이 d=440mm, "
    "설계 축력 Pu=3000kN, 설계 모멘트 Mu=200kN-m, 설계 전단력 Vu=150kN. "
    "모든 항목을 검토하고 종합 판정을 내려주세요."
)
print("\n" + review_result)

In [ ]:
# 검증 함수
def verify_exercise_1():
    """Exercise 1 결과를 검증한다."""
    checks = []

    # Check 1: review_result 존재
    checks.append('review_result' in globals() and review_result is not None)

    # Check 2: 도구가 5개 정의되었는지
    if 'structural_tools' in globals():
        checks.append(len(structural_tools) == 5)
    else:
        checks.append(False)

    # Check 3: 축력비 계산이 올바른지
    if 'execute_structural_tool' in globals():
        try:
            r = json.loads(execute_structural_tool(
                "calc_axial_ratio",
                {"Pu_kN": 3000, "fck_MPa": 27, "b_mm": 500, "h_mm": 500}
            ))
            # 3000*1000 / (0.85 * 27 * 250000) = 3000000 / 5737500 = 0.5229
            checks.append(0.50 < r["axial_ratio"] < 0.55)
        except Exception:
            checks.append(False)
    else:
        checks.append(False)

    # Check 4: 결과에 판정이 포함되어 있는지
    if 'review_result' in globals() and review_result:
        has_judgment = any(term in review_result for term in ["OK", "NG", "적합", "부적합", "판정"])
        checks.append(has_judgment)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_1()

---
## Exercise 2: 멀티에이전트 파이프라인 — Designer → Reviewer → Reporter

### 문제
3개의 전문 에이전트가 협업하는 구조 설계 검토 파이프라인을 구현하세요.

**Agent 1 — Designer (설계자):**
- 입력: 설계 조건 (하중, 재료, 용도)
- 출력: 구체적인 RC 부재 설계안 (단면, 배근, 재료)
- 시스템: "KDS 14 20 기준 RC 설계 전문가"

**Agent 2 — Reviewer (검토자):**
- 입력: Designer의 설계안
- 출력: 검토 결과 JSON `{"pass": bool, "checks": [...], "feedback": str}`
- 시스템: "구조 안전 검토 전문가. 엄격한 기준 적용."
- 부적합 시 피드백을 Designer에게 전달 (최대 2회 반복)

**Agent 3 — Reporter (보고서 작성자):**
- 입력: 최종 설계안 + 검토 결과
- 출력: 공식 구조 검토 보고서
- 시스템: "구조 보고서 작성 전문가"

### 테스트 입력
```
설계 조건:
- 용도: 15층 오피스 건물 1층 기둥
- 설계 축력: Pu = 4,500 kN
- 설계 모멘트: Mu = 350 kN·m
- 설계 전단력: Vu = 250 kN
- 내진설계범주: D
- 재료: fck = 35 MPa, fy = 400 MPa (SD400)
```

### 기대 출력
```
[Round 1] Designer → 설계안 생성
[Round 1] Reviewer → 검토 (PASS/FAIL)
  (FAIL이면: 피드백 → Round 2)
[Final] Reporter → 구조 검토 보고서
```

In [ ]:
# TODO: Designer → Reviewer → Reporter 멀티에이전트 파이프라인 구현

design_conditions = """
설계 조건:
- 용도: 15층 오피스 건물 1층 기둥
- 설계 축력: Pu = 4,500 kN
- 설계 모멘트: Mu = 350 kN·m
- 설계 전단력: Vu = 250 kN
- 내진설계범주: D
- 재료: fck = 35 MPa, fy = 400 MPa (SD400)
"""

# Agent 1: Designer
# YOUR CODE HERE

# Agent 2: Reviewer
# YOUR CODE HERE

# Agent 3: Reporter
# YOUR CODE HERE

# Pipeline
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

def llm_call(prompt: str, system: str = None, temperature: float = 0.2) -> str:
    params = {
        "model": model, "max_tokens": 2500, "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

# Agent 1: Designer
def designer_agent(conditions: str, feedback: str = None) -> str:
    system = (
        "당신은 KDS 14 20 기준 RC 구조물 설계 전문가입니다. "
        "경제적이면서 안전한 설계를 생성합니다. "
        "단면 크기, 콘크리트 강도, 배근 상세(주근, 띠철근)를 구체적으로 제시하세요."
    )
    prompt = f"다음 조건에 맞는 RC 기둥 설계안을 제시하라:\n{conditions}"
    if feedback:
        prompt += f"\n\n검토자 피드백 (반드시 반영):\n{feedback}"
    return llm_call(prompt, system=system, temperature=0.3)

# Agent 2: Reviewer
def reviewer_agent(design: str) -> dict:
    system = (
        "당신은 구조 안전 검토 전문가입니다. "
        "KDS 14 20 기준에 따라 설계의 적합성을 엄격하게 평가합니다. "
        "축력비, 철근비, 띠철근 간격, 내진 상세를 모두 검토하세요."
    )
    prompt = (
        f"다음 RC 기둥 설계안을 KDS 14 20 기준으로 검토하라.\n\n"
        f"설계안:\n{design}\n\n"
        f'JSON으로 답하라: {{"pass": true/false, "checks": [{{"item": str, "status": "OK/NG", "reason": str}}], "feedback": str}}\n'
        f"JSON만 출력하라."
    )
    response = client.messages.create(
        model=model, max_tokens=1500, temperature=0.0,
        system=system,
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "```json\n"}
        ],
        stop_sequences=["```"]
    )
    try:
        return json.loads(response.content[0].text.strip())
    except json.JSONDecodeError:
        return {"pass": False, "checks": [], "feedback": "검토 결과 파싱 오류"}

# Agent 3: Reporter
def reporter_agent(design: str, review: dict) -> str:
    system = (
        "당신은 구조 보고서 작성 전문가입니다. "
        "설계안과 검토 결과를 체계적인 공식 보고서로 정리합니다."
    )
    prompt = (
        f"다음 정보를 바탕으로 구조 검토 보고서를 작성하라.\n\n"
        f"설계안:\n{design}\n\n"
        f"검토 결과:\n{json.dumps(review, ensure_ascii=False, indent=2)}\n\n"
        f"보고서 포함: 1.부재 개요 2.검토 항목별 결과 표 3.종합 판정 4.권고사항"
    )
    return llm_call(prompt, system=system)

# === Pipeline ===
def design_review_pipeline(conditions: str, max_rounds: int = 2) -> str:
    feedback = None
    design = None
    review = None

    for round_num in range(1, max_rounds + 1):
        # Designer
        print(f"\n[Round {round_num}] Designer Agent...")
        design = designer_agent(conditions, feedback)
        print(f"  → 설계안 생성 ({len(design)}자)")

        # Reviewer
        print(f"[Round {round_num}] Reviewer Agent...")
        review = reviewer_agent(design)
        is_pass = review.get("pass", False)
        n_checks = len(review.get("checks", []))
        print(f"  → 검토 결과: {'PASS' if is_pass else 'FAIL'} ({n_checks}개 항목)")

        if is_pass:
            print(f"  → {round_num}회 만에 적합 판정!")
            break

        feedback = review.get("feedback", "개선이 필요합니다.")
        print(f"  → 피드백: {feedback[:100]}")

    # Reporter
    print(f"\n[Final] Reporter Agent...")
    report = reporter_agent(design, review)
    print(f"  → 보고서 생성 ({len(report)}자)")

    return report

# 실행
design_conditions = """
설계 조건:
- 용도: 15층 오피스 건물 1층 기둥
- 설계 축력: Pu = 4,500 kN
- 설계 모멘트: Mu = 350 kN·m
- 설계 전단력: Vu = 250 kN
- 내진설계범주: D
- 재료: fck = 35 MPa, fy = 400 MPa (SD400)
"""

pipeline_report = design_review_pipeline(design_conditions, max_rounds=2)
print("\n" + "="*60)
print(pipeline_report)

In [ ]:
# 검증 함수
def verify_exercise_2():
    """Exercise 2 결과를 검증한다."""
    checks = []

    # Check 1: pipeline_report 존재
    checks.append('pipeline_report' in globals() and pipeline_report is not None)

    # Check 2: 보고서가 충분한 길이
    if 'pipeline_report' in globals() and pipeline_report:
        checks.append(len(pipeline_report) > 200)
    else:
        checks.append(False)

    # Check 3: 3개 에이전트 함수가 정의되었는지
    funcs = ['designer_agent', 'reviewer_agent', 'reporter_agent']
    checks.append(all(f in globals() and callable(globals()[f]) for f in funcs))

    # Check 4: 보고서에 구조 관련 내용이 있는지
    if 'pipeline_report' in globals() and pipeline_report:
        has_structural = any(t in pipeline_report for t in ["기둥", "축력", "강도", "판정", "검토"])
        checks.append(has_structural)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_2()

---
## Exercise 3: 설계 최적화 에이전트 — Evaluator-Optimizer

### 문제
Evaluator-Optimizer 패턴으로 **RC 기둥 단면 최적화** 에이전트를 구현하세요.

**목표**: 주어진 하중 조건을 만족하면서 **가장 경제적인** (최소 단면) 기둥 설계를 찾는다.

**Generator (설계 생성기):**
- 하중 조건과 피드백을 받아 RC 기둥 설계를 제안
- JSON 형식: `{"b_mm": int, "h_mm": int, "fck_MPa": int, "rebar": str, "As_mm2": float}`

**Evaluator (설계 평가기):**
- 설계의 구조적 안전성을 **실제 계산**으로 검증 (도구 사용)
- 경제성 점수 = 100 - (단면적/최대단면적 * 50) - (철근량/최대철근량 * 50)
- JSON 형식: `{"safe": bool, "economy_score": float, "feedback": str}`

**최적화 루프:**
```
Round 1: 큰 단면으로 시작 → 안전하지만 비경제적
Round 2: 피드백에 따라 단면 축소 → 경제적이지만 안전성 확인
Round 3: 최적 균형점 찾기
```

### 테스트 입력
```
하중 조건:
- Pu = 3,500 kN
- Mu = 250 kN·m
- fck = 30 MPa, fy = 400 MPa
- 최대 축력비 한도: 0.65 (내진 고려)
```

### 기대 출력
```
[Round 1] 설계: 700x700, 12-D25 → 안전(O), 경제성 45점
  피드백: "단면이 과대. 600x600으로 축소 시도"
[Round 2] 설계: 600x600, 10-D25 → 안전(O), 경제성 65점
  피드백: "적절한 수준. 철근량 미세 조정 시도"
[Round 3] 설계: 600x600, 8-D29 → 안전(O), 경제성 72점
  → 최적 설계!
```

In [ ]:
# TODO: Evaluator-Optimizer 설계 최적화 에이전트 구현

load_conditions = {
    "Pu_kN": 3500,
    "Mu_kNm": 250,
    "fck_MPa": 30,
    "fy_MPa": 400,
    "max_axial_ratio": 0.65
}

# Generator (설계 생성기)
# YOUR CODE HERE

# Evaluator (설계 평가기) — 실제 계산 포함
# YOUR CODE HERE

# 최적화 루프
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

load_conditions = {
    "Pu_kN": 3500,
    "Mu_kNm": 250,
    "fck_MPa": 30,
    "fy_MPa": 400,
    "max_axial_ratio": 0.65
}

# 최대 단면 (경제성 기준)
MAX_SECTION = 800 * 800  # mm2
MAX_REBAR = 8000  # mm2

def generate_design(conditions: dict, feedback: str = None) -> dict:
    """하중 조건에 맞는 RC 기둥 설계를 생성한다."""
    prompt = (
        f"다음 하중 조건에 맞는 RC 기둥 설계를 JSON으로 제시하라.\n"
        f"조건: {json.dumps(conditions, ensure_ascii=False)}\n"
    )
    if feedback:
        prompt += f"이전 설계 피드백 (반드시 반영): {feedback}\n"
    prompt += (
        f'\nJSON 형식: {{"b_mm": int, "h_mm": int, "fck_MPa": int, '
        f'"rebar": "nD-XX", "As_mm2": float, "d_mm": float}}\n'
        f"JSON만 출력하라."
    )
    response = client.messages.create(
        model=model, max_tokens=500, temperature=0.3,
        system="RC 구조 설계 전문가. 경제적이면서 안전한 설계를 생성하라.",
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "```json\n"}
        ],
        stop_sequences=["```"]
    )
    try:
        return json.loads(response.content[0].text.strip())
    except json.JSONDecodeError:
        return {"b_mm": 600, "h_mm": 600, "fck_MPa": 30, "rebar": "8-D25", "As_mm2": 3973, "d_mm": 540}

def evaluate_design(design: dict, conditions: dict) -> dict:
    """설계의 안전성과 경제성을 평가한다."""
    b = design["b_mm"]
    h = design["h_mm"]
    fck = design.get("fck_MPa", conditions["fck_MPa"])
    fy = conditions["fy_MPa"]
    As = design["As_mm2"]
    d = design.get("d_mm", h - 60)
    Pu = conditions["Pu_kN"]
    Mu = conditions["Mu_kNm"]
    max_ratio = conditions["max_axial_ratio"]

    # 구조 검토
    Ag = b * h
    axial_ratio = (Pu * 1000) / (0.85 * fck * Ag)
    rebar_ratio = As / (b * d)
    a = (As * fy) / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn

    axial_ok = axial_ratio <= max_ratio
    rebar_ok = 0.01 <= rebar_ratio <= 0.08
    moment_ok = phi_Mn >= Mu
    safe = axial_ok and rebar_ok and moment_ok

    # 경제성 점수 (0~100, 높을수록 경제적)
    section_ratio = min(Ag / MAX_SECTION, 1.0)
    rebar_amount_ratio = min(As / MAX_REBAR, 1.0)
    economy_score = 100 - (section_ratio * 50) - (rebar_amount_ratio * 50)
    economy_score = max(0, round(economy_score, 1))

    # 피드백 생성
    feedback_parts = []
    if not axial_ok:
        feedback_parts.append(f"축력비 {axial_ratio:.3f} > {max_ratio}. 단면을 키우거나 fck를 높여라.")
    if not rebar_ok:
        feedback_parts.append(f"철근비 {rebar_ratio:.4f}가 범위(0.01~0.08) 밖이다.")
    if not moment_ok:
        feedback_parts.append(f"phi*Mn={phi_Mn:.1f} < Mu={Mu}. 철근량을 늘려라.")
    if safe and economy_score < 60:
        feedback_parts.append(f"안전하지만 비경제적(점수 {economy_score}). 단면이나 철근을 줄여봐라.")
    if safe and economy_score >= 60:
        feedback_parts.append("안전하고 경제적인 설계입니다.")

    return {
        "safe": safe,
        "economy_score": economy_score,
        "axial_ratio": round(axial_ratio, 4),
        "rebar_ratio": round(rebar_ratio, 5),
        "phi_Mn_kNm": round(phi_Mn, 1),
        "Mu_kNm": Mu,
        "checks": {
            "axial": "OK" if axial_ok else "NG",
            "rebar": "OK" if rebar_ok else "NG",
            "moment": "OK" if moment_ok else "NG"
        },
        "feedback": " ".join(feedback_parts)
    }

def optimize_design(conditions: dict, max_rounds: int = 4) -> dict:
    """Evaluator-Optimizer 루프로 설계를 최적화한다."""
    feedback = None
    best_design = None
    best_score = -1
    history = []

    for round_num in range(1, max_rounds + 1):
        # Generate
        design = generate_design(conditions, feedback)
        print(f"\n[Round {round_num}] 설계: {design.get('b_mm','?')}x{design.get('h_mm','?')}, "
              f"{design.get('rebar','?')} (As={design.get('As_mm2','?')}mm2)")

        # Evaluate
        evaluation = evaluate_design(design, conditions)
        safe = evaluation["safe"]
        score = evaluation["economy_score"]
        print(f"  안전: {'O' if safe else 'X'} | 경제성: {score}점")
        print(f"  축력비: {evaluation['axial_ratio']} ({evaluation['checks']['axial']})")
        print(f"  철근비: {evaluation['rebar_ratio']} ({evaluation['checks']['rebar']})")
        print(f"  phi*Mn: {evaluation['phi_Mn_kNm']} vs Mu: {evaluation['Mu_kNm']} ({evaluation['checks']['moment']})")

        history.append({"round": round_num, "design": design, "evaluation": evaluation})

        # Best 업데이트
        if safe and score > best_score:
            best_design = design
            best_score = score

        # 종료 조건: 안전하고 경제성 60점 이상
        if safe and score >= 60:
            print(f"\n[완료] {round_num}회 만에 최적 설계 도달! (경제성 {score}점)")
            break

        feedback = evaluation["feedback"]
        print(f"  피드백: {feedback}")

    return {
        "best_design": best_design,
        "best_score": best_score,
        "history": history,
        "rounds": len(history)
    }

# 실행
print("=== RC 기둥 설계 최적화 에이전트 ===")
optimization_result = optimize_design(load_conditions, max_rounds=4)

print(f"\n{'='*60}")
print(f"최적 설계: {json.dumps(optimization_result['best_design'], ensure_ascii=False, indent=2)}")
print(f"경제성 점수: {optimization_result['best_score']}")
print(f"총 라운드: {optimization_result['rounds']}")

In [ ]:
# 검증 함수
def verify_exercise_3():
    """Exercise 3 결과를 검증한다."""
    checks = []

    # Check 1: optimization_result 존재
    checks.append('optimization_result' in globals() and optimization_result is not None)

    # Check 2: best_design이 존재하는지
    if 'optimization_result' in globals() and optimization_result:
        checks.append(optimization_result.get('best_design') is not None)
    else:
        checks.append(False)

    # Check 3: evaluate_design이 올바르게 동작하는지
    if 'evaluate_design' in globals():
        try:
            test_design = {"b_mm": 600, "h_mm": 600, "fck_MPa": 30, "As_mm2": 3973, "d_mm": 540}
            test_eval = evaluate_design(test_design, load_conditions)
            checks.append('safe' in test_eval and 'economy_score' in test_eval)
        except Exception:
            checks.append(False)
    else:
        checks.append(False)

    # Check 4: history에 최소 1개 라운드가 있는지
    if 'optimization_result' in globals() and optimization_result:
        checks.append(len(optimization_result.get('history', [])) >= 1)
    else:
        checks.append(False)

    # Check 5: best_design이 안전한지
    if 'optimization_result' in globals() and optimization_result:
        bd = optimization_result.get('best_design')
        if bd:
            ev = evaluate_design(bd, load_conditions)
            checks.append(ev['safe'])
        else:
            checks.append(False)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_3()

---
## 종합 정리

| Exercise | 패턴 | 건축공학 적용 |
|---|---|---|
| Ex.1 | Agent Loop + Tools | RC 부재 자동 검토 (축력비, 철근비, 강도) |
| Ex.2 | Multi-Agent Pipeline | Designer → Reviewer → Reporter 협업 |
| Ex.3 | Evaluator-Optimizer | RC 기둥 단면 최적화 (안전성 + 경제성) |

### 핵심 학습 포인트
1. **에이전트 루프**의 핵심은 `while stop_reason == "tool_use"` 루프
2. **도구 설계**가 에이전트 성능의 핵심 — docstring, 타입, 에러 메시지 품질
3. **멀티에이전트**에서 각 에이전트의 역할과 시스템 프롬프트를 명확히 분리
4. **Evaluator-Optimizer**는 구조 설계 최적화에 자연스럽게 적용됨
5. 모든 패턴에 **Guardrails** (반복 제한, 검증)를 반드시 포함